## Working with NetCDF files

In [1]:
import pandas as pd
import netCDF4 as nc # pip install netCDF4
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
df = nc.Dataset('temp_anomalies/gistemp1200_GHCNv4_ERSSTv5.nc') # read in your netCDF file

In [4]:
# confirm the data type
df.data_model

'NETCDF3_CLASSIC'

### Accessing groups in NetCDF4 data

In [5]:
df

<class 'netCDF4.Dataset'>
root group (NETCDF3_CLASSIC data model, file format NETCDF3):
    title: GISTEMP Surface Temperature Analysis
    institution: NASA Goddard Institute for Space Studies
    source: http://data.giss.nasa.gov/gistemp/
    Conventions: CF-1.6
    history: Created 2025-07-08 19:16:36 by SBBX_to_nc 2.0 - ILAND=1200, IOCEAN=NCDC/ER5, Base: 1951-1980
    dimensions(sizes): lat(90), lon(180), time(1746), nv(2)
    variables(dimensions): float32 lat(lat), float32 lon(lon), int32 time(time), int32 time_bnds(time, nv), int16 tempanomaly(time, lat, lon)
    groups: 

## Access the data dictionary
The dictionary is _inside the data_.<br>
Pay attention to the 'missing_value' and ' _FillValue' fields.

In [6]:
df.variables # returns a dictionary on the variables in the dataset

{'lat': <class 'netCDF4.Variable'>
 float32 lat(lat)
     standard_name: latitude
     long_name: Latitude
     units: degrees_north
 unlimited dimensions: 
 current shape = (90,)
 filling on, default _FillValue of 9.969209968386869e+36 used,
 'lon': <class 'netCDF4.Variable'>
 float32 lon(lon)
     standard_name: longitude
     long_name: Longitude
     units: degrees_east
 unlimited dimensions: 
 current shape = (180,)
 filling on, default _FillValue of 9.969209968386869e+36 used,
 'time': <class 'netCDF4.Variable'>
 int32 time(time)
     long_name: time
     units: days since 1800-01-01 00:00:00
     bounds: time_bnds
 unlimited dimensions: 
 current shape = (1746,)
 filling on, default _FillValue of -2147483647 used,
 'time_bnds': <class 'netCDF4.Variable'>
 int32 time_bnds(time, nv)
 unlimited dimensions: 
 current shape = (1746, 2)
 filling on, default _FillValue of -2147483647 used,
 'tempanomaly': <class 'netCDF4.Variable'>
 int16 tempanomaly(time, lat, lon)
     long_name: Sur

In [7]:
df.variables['time'] # returns the time variable

<class 'netCDF4.Variable'>
int32 time(time)
    long_name: time
    units: days since 1800-01-01 00:00:00
    bounds: time_bnds
unlimited dimensions: 
current shape = (1746,)
filling on, default _FillValue of -2147483647 used

In [8]:
df.variables['tempanomaly'] # returns the anomaly variable

<class 'netCDF4.Variable'>
int16 tempanomaly(time, lat, lon)
    long_name: Surface temperature anomaly
    units: K
    scale_factor: 0.01
    cell_methods: time: mean
    _FillValue: 32767
unlimited dimensions: 
current shape = (1746, 90, 180)
filling on

In [9]:
# let's look at the last three values for "anomaly," since that's what we're interested in
# access them by index using a slice notation
df.variables['tempanomaly'][-3:].data # use '.data' to get information from this class

array([[[ 1.3399999,  1.3399999,  1.3399999, ...,  1.3399999,
          1.3399999,  1.3399999],
        [ 1.3399999,  1.3399999,  1.3399999, ...,  1.3399999,
          1.3399999,  1.3399999],
        [ 1.3399999,  1.3399999,  1.3399999, ...,  1.3399999,
          1.3399999,  1.3399999],
        ...,
        [ 1.64     ,  1.64     ,  1.64     , ...,  1.64     ,
          1.64     ,  1.64     ],
        [ 1.64     ,  1.64     ,  1.64     , ...,  1.64     ,
          1.64     ,  1.64     ],
        [ 1.64     ,  1.64     ,  1.64     , ...,  1.64     ,
          1.64     ,  1.64     ]],

       [[ 0.48     ,  0.48     ,  0.48     , ...,  0.48     ,
          0.48     ,  0.48     ],
        [ 0.48     ,  0.48     ,  0.48     , ...,  0.48     ,
          0.48     ,  0.48     ],
        [ 0.48     ,  0.48     ,  0.48     , ...,  0.48     ,
          0.48     ,  0.48     ],
        ...,
        [ 3.8      ,  3.8      ,  3.8      , ...,  3.8      ,
          3.8      ,  3.8      ],
        [ 3.

## What are we looking at?
There are thousands of raster datsets with information in them. These are stored as grids of data where the first value is the highest lat/lon value of temp anomalies at a certain time.

## Averaging data for 2024
We want the 12 temperature anomalies from 2024.<br>
To figure out which numbers to subset, we need to get information for the 'time' variable.<br>
The anomaly data will be in arrays.

## How to slice 🔪
https://bas.codes/posts/python-slicing

In [10]:
# what are we dealing with in this `time` variable?
# time to slice! 
# let's look at the first three values and last three values
print(df.variables['time'][:3].data)
print(df.variables['time'][-3:].data)

[29233 29264 29293]
[82284 82314 82345]


## But shouldn't the values start at zero?
Time is measured in days since 1-1-1800

In [11]:
df.variables['time']

<class 'netCDF4.Variable'>
int32 time(time)
    long_name: time
    units: days since 1800-01-01 00:00:00
    bounds: time_bnds
unlimited dimensions: 
current shape = (1746,)
filling on, default _FillValue of -2147483647 used

## So where are we starting?

In [12]:
first = df.variables['time'][0].data
print(first)

29233


In [14]:
# okay, the first element in the data is 29233
# the units are days since 1/1/1800, so we can do some napkin math
_years = first / 365
print(_years)
print(1800 + _years)

80.09041095890412
1880.0904109589042


In [15]:
# and where do we actually end with data?
last = df.variables['time'][-1].data
print(last)
print(1800 + last / 365)

82345
2025.6027397260275


In [18]:
# now lets get more specific
from datetime import date
from datetime import timedelta

start_date = date(1800, 1, 1)
last_date = start_date + timedelta(days=last)

TypeError: unsupported type for timedelta days component: numpy.ndarray

In [19]:
# oh no
type(last)

numpy.ndarray

In [20]:
# that's surprising. it really looked like it was just a number
last.size

1

In [21]:
# well, okay, we can just grab that one index and then we'll probably have a number

# https://numpy.org/doc/2.2/reference/generated/numpy.ndarray.flat.html#numpy.ndarray.flat
usable_last = last.flat[0]

In [22]:
# let's try again
last_date = start_date + timedelta(days=usable_last)

TypeError: unsupported type for timedelta days component: numpy.int32

In [23]:
# we can't use a numpy.float data type
type(usable_last)

numpy.int32

In [24]:
# try converting to an int using pandas
actually_usable_last = int(usable_last)
type(actually_usable_last)

int

In [25]:
last_date = start_date + timedelta(days=actually_usable_last)
print(last_date)

2025-06-15


In [26]:
# just for fun, find the date of the first data entry in this data set
usable_first = int(first.flat[0])
first_date = start_date + timedelta(days=usable_first)
first_date

datetime.date(1880, 1, 15)

# 🎉 🎉 🎉

## Find all the months in 2024
We're using monthly data, so we want to find data for all of the months that fall between Jan. 1, 2024 and Dec. 31, 2024. <br>
We'll define that range, then clamp the data to return those 12 months.

In [27]:
# define variables for the start date and end date that we want
start_2024 = date(2024, 1, 1)
end_2024 = date(2024, 12, 31)

In [28]:
# figure out how many days it has been between when we started counting, on Jan. 1, 1800 to Jan. 1, 2024. 
# we're finding the value that corresponds with Jan. 1, 2024
# since the time measurement counts up, we're starting with the larger number and subtracting
range_min = start_2024 - start_date
range_min

datetime.timedelta(days=81814)

In [29]:
# same thing for our end date
range_max = end_2024 - start_date
range_max

datetime.timedelta(days=82179)

In [30]:
# as we learned above, let's not get tripped up by passing around non-integer values
rmin = range_min.days
print(rmin, type(rmin))

rmax = range_max.days
print(rmax, type(rmax))

81814 <class 'int'>
82179 <class 'int'>


In [31]:
# get all of the values stored in `time` by using an empty slice, which is a copy
time = df.variables['time'][:]

# numpy will let us filter those down to just the ones in our range
indices = np.where((time >= rmin) & (time <= rmax))[0]

print(time)
print(indices)

[29233 29264 29293 ... 82284 82314 82345]
[1728 1729 1730 1731 1732 1733 1734 1735 1736 1737 1738 1739]


In [32]:
# this makes sense. wherever there is data, there is an index
# it's a representation of where the data is in the file
df.variables['time'].shape

(1746,)

In [33]:
# now we can use those index numbers to get data in just the window we wanted for time,
# these numbers will look familiar based on the bounds we set above

# this is akin to more familiar indexing you've used like `var[0]` or `var[14]`
# but pandas lets us pass an array of individual numbers all at once, which is what
# our variable `indices` is 

# basically, we're getting time[1728] and time[1729] ... through time[1739]
df.variables['time'][indices]

masked_array(data=[81828, 81859, 81888, 81919, 81949, 81980, 82010, 82041,
                   82072, 82102, 82133, 82163],
             mask=False,
       fill_value=np.int64(999999),
            dtype=int32)

In [34]:
# what will get more interesting is when we use these index numbers to check out the other netcdf variables

# first, we can sanity check that the shape of the time variable and the shape of the anomaly variable match
print(df.variables['time'].shape)
print(df.variables['tempanomaly'].shape)

(1746,)
(1746, 90, 180)


In [35]:
df.variables['tempanomaly']

<class 'netCDF4.Variable'>
int16 tempanomaly(time, lat, lon)
    long_name: Surface temperature anomaly
    units: K
    scale_factor: 0.01
    cell_methods: time: mean
    _FillValue: 32767
unlimited dimensions: 
current shape = (1746, 90, 180)
filling on

In [36]:
# the `anom` coordinates are: time lat lon
# our `indices` variable works on the `time` coordinate, so we'll explicitly slice `anom` in the right spot,
# using empty slice notation for the other coordinates to leave them intact
slice_2024 = df.variables['tempanomaly'][indices,:,:]

# this happens to have the same result as:

#     df.variables['anom'][indices]

# but the explicit way is a better approach for when your data might not magically work out

slice_2024.shape

(12, 90, 180)

### A quick diversion about masked values

In [37]:
# this distinction will be important in a moment
print(type(slice_2024))
print(type(slice_2024.data))

<class 'numpy.ma.MaskedArray'>
<class 'numpy.ndarray'>


In [38]:
# here's a cool trick:
slice_2024.mask

array([[[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, Fal

This will give you a giant array of booleans indicating where in your data all the masked values live, just waiting to make your day worse.

In [39]:
# check for masked values (missing or filled values)
if isinstance(slice_2024, np.ma.MaskedArray):
    # The .mask attribute is a boolean array, so it can be summed with numpy
    mask_count = np.sum(slice_2024.mask)
    print(mask_count)

1751


In [40]:
# netcdf4 files *usually* have attributes expressly defining the mask/fill values within them:
if '_FillValue' in df.variables['tempanomaly'].ncattrs():
    mask_fill = df.variables['tempanomaly']._FillValue
    print('Fill Value', mask_fill)

Fill Value 32767


In [41]:
# attribute names are semi-arbitrary, though, so there's a built-in way that also handles
# some less common ways of defining missing data

df.variables['tempanomaly'].get_fill_value()

np.int16(32767)

In [42]:
# that's a big number
# I wonder if that will make our data unreasonable
slice_2024.data.mean()

np.float32(296.5613)

I don't know if we can publish a 300 degree temperature anomaly.

In [43]:
slice_2024.mean()

np.float64(1.4352523890598965)

That feels better.

But why are those numbers so different?

Remember when I said the distinction between the two types in the first cell of this section were going to matter?

Basically, the ndarray still includes all the non-filled values. They can't be zeroes, because a zero is also a number, and is a reasonable measurement that we might actually find. Numpy has a type for this, which is `np.nan` (which stands for Not-a-Number), but the netCDF file doesn't have a way to encode this into its data structure. Or, rather, the way it has is to define some other thing and then tell you about it, which is what we encountered with the fill values earlier.

The MaskedArray is able to account for these masked values, and running its `mean()` method automatically excludes them from the math.

In [44]:
# numpy has a nanmean() function which also works that way
# we don't need to use it for this file, as discussed above,
# but we can better illustrate what happened above if you understand how it works

multi_dimensional_data = np.array([[7, np.nan, 3], [np.nan, 5, 1]])

print(multi_dimensional_data)

print() # newlines are nice for taking in information

print('Shape:', multi_dimensional_data.shape)

# this example is only 2-dimensional so it's simple to look at
# here we'll actually calculate the means for each dimension
mean_axis0 = np.nanmean(multi_dimensional_data, axis=0)
mean_axis1 = np.nanmean(multi_dimensional_data, axis=1)

[[ 7. nan  3.]
 [nan  5.  1.]]

Shape: (2, 3)


In [45]:
# what do you think the values of those two means are?

# here's a hint:
print(mean_axis0.shape, mean_axis1.shape)

(3,) (2,)


In [46]:
# pencils down
print("Mean across axis 0 (columns):", mean_axis0)
print("Mean across axis 1 (rows):", mean_axis1)

# does this make sense? we can take a second to discuss these results.... 

Mean across axis 0 (columns): [7. 5. 2.]
Mean across axis 1 (rows): [5. 3.]


### Back to our example.

Now that we have the months we want, we need to average the data.

In [47]:
# remember, our slice is
#  12 months of temperature measurements
#  a latitude
#  a longitude
slice_2024.data.shape

(12, 90, 180)

Which axis do you think we need to use for our average?

In [ ]:
# cool. let's go get our numbers
average_anomalies_2024 = slice_2024.mean(axis=0)

In [ ]:
print(type(average_anomalies_2024))

# sanity check
average_anomalies_2024.shape

### Let's export our analysis so we can map it 📸
`brew install gdal`, if you don't already have it, then `pip install gdal`

Now we want to export the data to a geotiff! Data in geotiffs are stored in bands. When we're talking about
data that goes in bands we're talking about the non-coordinate data. The values that measure something in each
raster cell. We'll use the lat/lon arrays to tell QGIS where to position each of these raster cells.
The geotiff does not store the lat/long for each cell value though. It just needs to know the resolution
and where to start/stop the grid. We can calculate this from our input data!

In [ ]:
# osgeo is (confusingly) the package name for gdal when importing
# if you try to `pip install osgeo`, it will recommend you install `gdal` instead
from osgeo import gdal, osr

# don't get too hung up on the specifics of this just yet...
# it's called from below and the variables it operates on aren't set up until the next couple cells
# the necessary return values are defined here: https://gdal.org/en/stable/tutorials/geotransforms_tut.html
def getGeoTransform(extent, ncols, nlines):
    resx = (extent[2] - extent[0]) / ncols
    resy = (extent[3] - extent[1]) / nlines

    # as a reminder from the link above, the order of values here for a North-Up image are:
    
    # GT(2), GT(4) just zero
    # GT(1), GT(5) the pixel size
    # GT(0), GT(3) top left corner of the top left pixel

    return [extent[0], resx, 0, extent[3] , 0, -resy]

In [ ]:
# okay, we need to know the geographic extent of our image/data

# here we return to referencing our original dataframe for ease
# and because we only filtered by date, the coordinates should still be accurate/the same
lats = df.variables['lat'][:].data
lons = df.variables['lon'][:].data
 
# we need to use the extent of the data more than once in the `GeoTransform()` function above,
# so we'll pre-compute the ranges and hold onto them as an indexed array:
# [top-left lon, top-left lat, bottom-right lon, bottom-right lat]
extent = [lons.min(),lats.max(),lons.max(),lats.min()]
# that looks like a lot, but you can picture it in your head:

# latitude is north-south, or Y-axis
# longitude is east-west, or X-axis

# so the top-left would be the smallest X-axis (lon) and the largest Y axis (lat)
# and the bottom-right reverses it
# yeah?

In [ ]:
# finally, we'll stash some more details about our data
# (these will mostly be related to the dimensions of our output image)
num_lines = average_anomalies_2024.shape[0]
num_cols = average_anomalies_2024.shape[1]
num_bands = 1 # we only have one measurement to encode into the file, so `average_anomalies_2024.shape[0]` feels like overkill
data_type = gdal.GDT_Float32

In [ ]:
# now we can actually create the GeoTiff:

# tell gdal what we want
driver = gdal.GetDriverByName('GTiff')

# instantiate an empty grid with our dimensions
grid_data = driver.Create('grid_data', num_cols, num_lines, num_bands, data_type)

# pass in the data to our bands
grid_data.GetRasterBand(1).WriteArray(average_anomalies_2024)
 
# project our data using the WGS84 Spatial Reference System
srs = osr.SpatialReference()
srs.ImportFromProj4('+proj=longlat +ellps=WGS84 +datum=WGS84 +no_defs')
 
# apply the projection and geo-transform
grid_data.SetProjection(srs.ExportToWkt())

# this line calls our function above to get the magic coefficients for an affine transformation
grid_data.SetGeoTransform(getGeoTransform(extent, num_cols, num_lines))
# (now that we know what these parameters are for our `getGeoTransform` function, scroll back to it and try to puzzle it out)
 
# copy the newly populated and transformed file to something more recognizable
file_name = 'average_anomalies_2024.tif'
driver.CreateCopy(file_name, grid_data, 0)
print(f'Generated GeoTIFF: {file_name}')

# close the files to clean-up memory
driver = None
grid_data = None
 
# delete the temp file
import os                
os.remove('grid_data')

## Let's check our work
Pull the GeoTIFF into QGIS, apply a new color scale, and check it against another published map:<br>
https://berkeleyearth.org/global-temperature-report-for-2024/